In [1]:
# phase_1_logistic_baseline.py

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
import optuna
import logging
import time
import os
import pickle
from scipy.stats import linregress
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

# =============================================================================
# SCRIPT CONFIGURATION
# =============================================================================

# -- File & Path Configuration --
HDF_FILE_PATH = '../../data/raw/all_hourly_data.h5'
OUTPUT_DIR = 'phase_1_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -- Experiment Configuration --
# Set to True to run on a small sample for fast testing of the pipeline
DRY_RUN = False
DRY_RUN_PATIENTS = 1000  # Number of patients to use in dry run mode (increased for better stratification)

# Set to True to use cached preprocessed data if available (saves significant time)
USE_CACHED_PREPROCESSING = True  # Temporarily disabled to regenerate features with fixes

# Feature engineering options
CALCULATE_TRENDS = True  # Set to False to disable trend calculations (may improve performance)

# Time window configuration
WINDOW_SIZE = 24  # Hours of data to use for feature extraction
GAP_TIME = 6      # Hours of gap before prediction window (to avoid data leakage)

# Prediction target from the 'patients' table in MIMIC-Extract
TARGET_VARIABLE = 'mort_hosp'

# -- Hyperparameter Tuning Configuration --
N_OPTUNA_TRIALS = 15  # Number of trials for hyperparameter search
OPTUNA_TIMEOUT = 1800 # Timeout for the study in seconds (e.g., 1 hour)

# -- Reproducibility --
SEED = 42

# -- Data Splitting Strategy --
# IMPORTANT: We split by subject_id (not icustay_id) to prevent data leakage
# This ensures no patient appears in both training and test sets, even with multiple ICU stays

# =============================================================================
# LOGGING SETUP
# =============================================================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] - %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(OUTPUT_DIR, 'phase_1_log.txt'), mode='w'),
        logging.StreamHandler()
    ]
)

# =============================================================================
# FEATURE DEFINITION
# =============================================================================

# List of features to be treated as dynamic, based on your provided list
DYNAMIC_FEATURES = [
    'diastolic blood pressure', 'heart rate', 'mean blood pressure', 'oxygen saturation',
    'respiratory rate', 'systolic blood pressure', 'temperature', 'fraction inspired oxygen',
    'fraction inspired oxygen set', 'peak inspiratory pressure', 'plateau pressure',
    'positive end-expiratory pressure', 'positive end-expiratory pressure set',
    'respiratory rate set', 'tidal volume observed', 'tidal volume set',
    'tidal volume spontaneous', 'cardiac index', 'cardiac output thermodilution',
    'cardiac output fick', 'central venous pressure', 'glascow coma scale total',
    'pulmonary artery pressure mean', 'pulmonary artery pressure systolic',
    'pulmonary capillary wedge pressure', 'systemic vascular resistance', 'anion gap',
    'bicarbonate', 'co2', 'co2 (etco2, pco2, etc.)', 'lactate', 'lactic acid',
    'partial pressure of carbon dioxide', 'partial pressure of oxygen', 'ph',
    'venous pvo2', 'blood urea nitrogen', 'calcium ionized', 'creatinine', 'glucose',
    'potassium', 'potassium serum', 'sodium', 'chloride', 'hematocrit', 'hemoglobin',
    'platelets', 'red blood cell count', 'fibrinogen', 'partial thromboplastin time',
    'prothrombin time inr', 'prothrombin time pt', 'troponin-i', 'troponin-t'
]

# =============================================================================
# DATA LOADING FUNCTION
# =============================================================================

def load_data(hdf_path, n_samples=None):
    """Loads patient and time-series data from the MIMIC-Extract HDF5 store."""
    logging.info(f"Starting data loading from: {hdf_path}")
    if not os.path.exists(hdf_path):
        logging.error(f"CRITICAL ERROR: File not found at '{hdf_path}'")
        raise FileNotFoundError(f"Data file not found: {hdf_path}")

    with pd.HDFStore(hdf_path, 'r') as store:
        logging.info("Loading patient data...")
        df_patients = store['/patients']
        logging.info(f"✓ Patients loaded: {df_patients.shape}")

        if n_samples:
            logging.warning(f"--- DRY RUN MODE: Sampling {n_samples} patients ---")
            all_patient_ids = df_patients.index.get_level_values('icustay_id').unique()
            np.random.seed(SEED)
            sampled_ids = np.random.choice(all_patient_ids, n_samples, replace=False)
            
            # Filter patients first
            df_patients = df_patients[df_patients.index.get_level_values('icustay_id').isin(sampled_ids)]
            logging.info(f"✓ Sampled patients: {df_patients.shape[0]}")
            
            # For dry run, load and filter in chunks to manage memory
            logging.info("Loading time-series data in chunks (Fixed format limitation)...")
            
            # Try to load in chunks to manage memory better
            try:
                # Load in smaller chunks if possible
                chunk_size = 100000  # Adjust based on available memory
                df_ts_chunks = []
                sampled_ids_set = set(sampled_ids)
                
                # Read the data in chunks and filter immediately
                total_rows = 0
                for chunk in pd.read_hdf(hdf_path, '/vitals_labs_mean', chunksize=chunk_size):
                    # Filter chunk immediately
                    filtered_chunk = chunk[chunk.index.get_level_values('icustay_id').isin(sampled_ids_set)]
                    if not filtered_chunk.empty:
                        df_ts_chunks.append(filtered_chunk)
                        total_rows += len(filtered_chunk)
                    
                    # Log progress
                    if len(df_ts_chunks) % 10 == 0:
                        logging.info(f"Processed {len(df_ts_chunks)} chunks, {total_rows} relevant rows found")
                
                # Combine filtered chunks
                if df_ts_chunks:
                    df_ts = pd.concat(df_ts_chunks, ignore_index=False)
                    logging.info(f"✓ Time-series loaded and filtered: {df_ts.shape}")
                else:
                    raise ValueError("No data found for sampled patients")
                    
            except Exception as e:
                # Fallback to loading full dataset if chunking fails
                logging.warning(f"Chunked loading failed ({e}), falling back to full load...")
                df_ts = store['/vitals_labs_mean']
                logging.info(f"✓ Full time-series loaded: {df_ts.shape}")
                
                # Filter to sampled patients immediately to free memory
                logging.info("Filtering to sampled patients...")
                sampled_ids_set = set(sampled_ids)
                df_ts = df_ts[df_ts.index.get_level_values('icustay_id').isin(sampled_ids_set)]
                logging.info(f"✓ Time-series filtered to sampled patients: {df_ts.shape}")
        else:
            logging.info("Loading full time-series data (vitals_labs_mean)...")
            df_ts = store['/vitals_labs_mean']
            logging.info(f"✓ Full time-series loaded: {df_ts.shape}")
        
        # Handle MultiIndex columns
        if isinstance(df_ts.columns, pd.MultiIndex):
            logging.info("Found MultiIndex, flattening to first level.")
            df_ts.columns = [col[0] for col in df_ts.columns]
        
        # Ensure there are no duplicate columns after flattening
        df_ts = df_ts.loc[:,~df_ts.columns.duplicated()]
        logging.info(f"✓ Time-series processed: {df_ts.shape}")

    return df_patients, df_ts

# =============================================================================
# FEATURE ENGINEERING FUNCTIONS
# =============================================================================

def _calculate_slope(series):
    """Helper to calculate slope of a time series, handling NaNs."""
    y = series.dropna()
    if len(y) < 2:
        return np.nan
    
    # Extract hours_in values from MultiIndex for x-coordinates
    if isinstance(y.index, pd.MultiIndex):
        x = y.index.get_level_values('hours_in').values
    else:
        x = y.index.values
    
    try:
        return linregress(x, y.values).slope
    except Exception:
        return np.nan

def engineer_features(df_ts, dynamic_cols, static_cols):
    """
    Engineers features from the time-series data for each patient stay.
    
    Args:
        df_ts (pd.DataFrame): Time-series data with 'icustay_id' and 'hours_in' index.
        dynamic_cols (list): List of columns to apply dynamic feature engineering.
        static_cols (list): List of columns to apply static feature engineering.

    Returns:
        pd.DataFrame: A dataframe with one row per icustay_id and engineered features.
    """
    logging.info(f"Starting feature engineering for {df_ts.index.get_level_values('icustay_id').nunique()} stays...")
    
    # Ensure index is sorted for trend calculations
    df_ts = df_ts.sort_index()
    
    # --- Process Dynamic Features ---
    logging.info(f"Processing {len(dynamic_cols)} dynamic features...")
    
    # Group by patient stay
    grouped = df_ts[dynamic_cols].groupby('icustay_id')
    
    # Basic aggregations - CRITICAL: Include 'mean' which was in the high-performing model
    # Note: 'last' now means last within the WINDOW_SIZE period (not overall last)
    dynamic_agg_basic = grouped.agg(['mean', 'std', 'min', 'max', 'last']).rename(columns={'last': f'last_in_{WINDOW_SIZE}h'})
    
    # Optional trend calculations (can be slow and may hurt performance)
    if CALCULATE_TRENDS:
        logging.info("Calculating 6-hour and 24-hour trends...")
        
        # Use a dictionary to store results to avoid performance warnings
        trend_results = {}
        
        for col in tqdm(dynamic_cols, desc="Calculating Slopes"):
            # 24-hour trend
            trend_results[f'{col}_trend_24h'] = grouped[col].apply(_calculate_slope)
            
            # 6-hour trend
            last_6h = df_ts[df_ts.index.get_level_values('hours_in') >= 18][col]
            if not last_6h.empty:
                trend_results[f'{col}_trend_6h'] = last_6h.groupby('icustay_id').apply(_calculate_slope)
            else:
                trend_results[f'{col}_trend_6h'] = pd.Series(np.nan, index=grouped.groups.keys())

        dynamic_agg_trends = pd.DataFrame(trend_results)
        # Combine dynamic features
        dynamic_features_df = pd.concat([dynamic_agg_basic, dynamic_agg_trends], axis=1)
    else:
        logging.info("Skipping trend calculations (CALCULATE_TRENDS=False)")
        dynamic_features_df = dynamic_agg_basic
    
    # Flatten multi-index columns properly (handle both tuples and strings)
    dynamic_features_df.columns = [
        '_'.join(col).strip() if isinstance(col, tuple) else col 
        for col in dynamic_features_df.columns.values
    ]
    
    # --- Process Static Features ---
    logging.info(f"Processing {len(static_cols)} static features...")
    static_features_df = df_ts[static_cols].groupby('icustay_id').agg(['mean', 'std'])
    static_features_df.columns = [
        '_'.join(col).strip() if isinstance(col, tuple) else col 
        for col in static_features_df.columns.values
    ]
    
    # --- Combine All Features ---
    logging.info("Combining all engineered features...")
    final_df = pd.concat([dynamic_features_df, static_features_df], axis=1)
    
    logging.info(f"✓ Feature engineering complete. Final shape: {final_df.shape}")
    return final_df

# =============================================================================
# PREPROCESSING CACHE FUNCTIONS
# =============================================================================

def get_cache_filename_prefix():
    """Generate a unique prefix for cache files based on experiment settings."""
    prefix = f"preprocessed_{TARGET_VARIABLE}"
    if DRY_RUN:
        prefix += f"_dryrun_{DRY_RUN_PATIENTS}"
    prefix += f"_trends_{CALCULATE_TRENDS}"
    prefix += f"_window_{WINDOW_SIZE}_gap_{GAP_TIME}"
    prefix += f"_seed_{SEED}"
    return prefix

def save_preprocessed_data(X_train, X_val, X_test, y_train, y_val, y_test, scaler, imputation_values):
    """Save preprocessed data to cache files."""
    prefix = get_cache_filename_prefix()
    
    cache_files = {
        'X_train': os.path.join(OUTPUT_DIR, f'{prefix}_X_train.pkl'),
        'X_val': os.path.join(OUTPUT_DIR, f'{prefix}_X_val.pkl'),
        'X_test': os.path.join(OUTPUT_DIR, f'{prefix}_X_test.pkl'),
        'y_train': os.path.join(OUTPUT_DIR, f'{prefix}_y_train.pkl'),
        'y_val': os.path.join(OUTPUT_DIR, f'{prefix}_y_val.pkl'),
        'y_test': os.path.join(OUTPUT_DIR, f'{prefix}_y_test.pkl'),
        'scaler': os.path.join(OUTPUT_DIR, f'{prefix}_scaler.pkl'),
        'imputation_values': os.path.join(OUTPUT_DIR, f'{prefix}_imputation_values.pkl')
    }
    
    logging.info("Saving preprocessed data to cache...")
    with open(cache_files['X_train'], 'wb') as f:
        pickle.dump(X_train, f)
    with open(cache_files['X_val'], 'wb') as f:
        pickle.dump(X_val, f)
    with open(cache_files['X_test'], 'wb') as f:
        pickle.dump(X_test, f)
    with open(cache_files['y_train'], 'wb') as f:
        pickle.dump(y_train, f)
    with open(cache_files['y_val'], 'wb') as f:
        pickle.dump(y_val, f)
    with open(cache_files['y_test'], 'wb') as f:
        pickle.dump(y_test, f)
    with open(cache_files['scaler'], 'wb') as f:
        pickle.dump(scaler, f)
    with open(cache_files['imputation_values'], 'wb') as f:
        pickle.dump(imputation_values, f)
    
    logging.info(f"✓ Preprocessed data cached with prefix: {prefix}")
    return cache_files

def load_preprocessed_data():
    """Load preprocessed data from cache files if they exist."""
    prefix = get_cache_filename_prefix()
    
    cache_files = {
        'X_train': os.path.join(OUTPUT_DIR, f'{prefix}_X_train.pkl'),
        'X_val': os.path.join(OUTPUT_DIR, f'{prefix}_X_val.pkl'),
        'X_test': os.path.join(OUTPUT_DIR, f'{prefix}_X_test.pkl'),
        'y_train': os.path.join(OUTPUT_DIR, f'{prefix}_y_train.pkl'),
        'y_val': os.path.join(OUTPUT_DIR, f'{prefix}_y_val.pkl'),
        'y_test': os.path.join(OUTPUT_DIR, f'{prefix}_y_test.pkl'),
        'scaler': os.path.join(OUTPUT_DIR, f'{prefix}_scaler.pkl'),
        'imputation_values': os.path.join(OUTPUT_DIR, f'{prefix}_imputation_values.pkl')
    }
    
    # Check if all cache files exist
    if not all(os.path.exists(f) for f in cache_files.values()):
        return None
    
    logging.info("Loading preprocessed data from cache...")
    try:
        with open(cache_files['X_train'], 'rb') as f:
            X_train = pickle.load(f)
        with open(cache_files['X_val'], 'rb') as f:
            X_val = pickle.load(f)
        with open(cache_files['X_test'], 'rb') as f:
            X_test = pickle.load(f)
        with open(cache_files['y_train'], 'rb') as f:
            y_train = pickle.load(f)
        with open(cache_files['y_val'], 'rb') as f:
            y_val = pickle.load(f)
        with open(cache_files['y_test'], 'rb') as f:
            y_test = pickle.load(f)
        with open(cache_files['scaler'], 'rb') as f:
            scaler = pickle.load(f)
        with open(cache_files['imputation_values'], 'rb') as f:
            imputation_values = pickle.load(f)
        
        logging.info(f"✓ Successfully loaded cached preprocessed data with prefix: {prefix}")
        logging.info(f"Data shapes: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")
        
        return X_train, X_val, X_test, y_train, y_val, y_test, scaler, imputation_values
    
    except Exception as e:
        logging.warning(f"Failed to load cached data: {e}")
        return None

# =============================================================================
# UNCERTAINTY QUANTIFICATION FUNCTIONS
# =============================================================================

def calculate_bootstrap_ci(y_true, y_pred_proba, metric_func, n_bootstrap=1000, confidence_level=0.95, random_state=42):
    """
    Calculate bootstrap confidence intervals for a given metric.
    
    Args:
        y_true: True binary labels
        y_pred_proba: Predicted probabilities 
        metric_func: Function to calculate metric (e.g., roc_auc_score)
        n_bootstrap: Number of bootstrap samples
        confidence_level: Confidence level (e.g., 0.95 for 95% CI)
        random_state: Random state for reproducibility
    
    Returns:
        dict: Contains 'point_estimate', 'ci_lower', 'ci_upper', 'std', 'bootstrap_scores'
    """
    np.random.seed(random_state)
    bootstrap_scores = []
    
    n_samples = len(y_true)
    
    # Perform bootstrap resampling
    for i in tqdm(range(n_bootstrap), desc="Bootstrap sampling"):
        # Stratified bootstrap to maintain class balance
        # Get indices for each class
        pos_indices = np.where(y_true == 1)[0]
        neg_indices = np.where(y_true == 0)[0]
        
        # Sample with replacement maintaining original class distribution
        n_pos = len(pos_indices)
        n_neg = len(neg_indices)
        
        boot_pos_indices = np.random.choice(pos_indices, size=n_pos, replace=True)
        boot_neg_indices = np.random.choice(neg_indices, size=n_neg, replace=True)
        boot_indices = np.concatenate([boot_pos_indices, boot_neg_indices])
        
        # Calculate metric on bootstrap sample
        try:
            score = metric_func(y_true[boot_indices], y_pred_proba[boot_indices])
            bootstrap_scores.append(score)
        except Exception as e:
            # Skip if bootstrap sample has issues (e.g., only one class)
            continue
    
    bootstrap_scores = np.array(bootstrap_scores)
    
    # Calculate confidence interval
    alpha = 1 - confidence_level
    lower_percentile = (alpha/2) * 100
    upper_percentile = (1 - alpha/2) * 100
    
    ci_lower = np.percentile(bootstrap_scores, lower_percentile)
    ci_upper = np.percentile(bootstrap_scores, upper_percentile)
    
    # Point estimate on original data
    point_estimate = metric_func(y_true, y_pred_proba)
    
    return {
        'point_estimate': point_estimate,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'std': np.std(bootstrap_scores),
        'bootstrap_scores': bootstrap_scores,
        'n_bootstrap': len(bootstrap_scores)
    }

def evaluate_with_uncertainty(y_true, y_pred_proba, y_pred=None, n_bootstrap=1000):
    """
    Comprehensive evaluation with uncertainty quantification.
    
    Args:
        y_true: True binary labels
        y_pred_proba: Predicted probabilities
        y_pred: Predicted binary labels (optional, will be calculated if None)
        n_bootstrap: Number of bootstrap samples
    
    Returns:
        dict: Comprehensive results with confidence intervals
    """
    if y_pred is None:
        y_pred = (y_pred_proba >= 0.5).astype(int)
    
    logging.info(f"Calculating bootstrap confidence intervals with {n_bootstrap} samples...")
    
    # Calculate AUROC with CI
    auroc_results = calculate_bootstrap_ci(y_true, y_pred_proba, roc_auc_score, n_bootstrap)
    
    # Calculate AUPRC with CI  
    auprc_results = calculate_bootstrap_ci(y_true, y_pred_proba, average_precision_score, n_bootstrap)
    
    # Basic metrics without CI (for completeness)
    basic_metrics = {
        'confusion_matrix': confusion_matrix(y_true, y_pred),
        'classification_report': classification_report(y_true, y_pred, output_dict=True)
    }
    
    return {
        'auroc': auroc_results,
        'auprc': auprc_results,
        'basic_metrics': basic_metrics
    }

# =============================================================================
# MODEL TUNING AND TRAINING
# =============================================================================

def tune_logistic_regression_with_optuna(X_train, y_train, X_val, y_val):
    """
    Uses Optuna to find the best hyperparameters for the Logistic Regression model.
    """
    logging.info(f"Starting Optuna hyperparameter search with {N_OPTUNA_TRIALS} trials...")
    
    # Calculate class weights for handling class imbalance
    classes = np.unique(y_train)
    class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weight_dict = {classes[i]: class_weights[i] for i in range(len(classes))}
    logging.info(f"Calculated class weights: {class_weight_dict}")

    def objective(trial):
        # Suggest hyperparameters for Logistic Regression
        C = trial.suggest_float('C', 1e-4, 1e2, log=True)
        penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet'])
        solver = trial.suggest_categorical('solver', ['liblinear', 'lbfgs', 'saga', 'sag'])
        
        # Define valid combinations of penalty and solver
        valid_combinations = {
            'l1': ['liblinear', 'saga'],
            'l2': ['liblinear', 'lbfgs', 'saga', 'sag'],
            'elasticnet': ['saga']
        }
        
        # Check if the combination is valid
        if solver not in valid_combinations[penalty]:
            # Return low score for invalid combinations
            return 0.5
        
        params = {
            'C': C,
            'penalty': penalty,
            'solver': solver,
            'random_state': SEED,
            'max_iter': 1000,
            'class_weight': class_weight_dict,
            'n_jobs': -1
        }
        
        # Add l1_ratio for elasticnet penalty
        if penalty == 'elasticnet':
            params['l1_ratio'] = trial.suggest_float('l1_ratio', 0.0, 1.0)
        
        try:
            # Additional data validation
            if np.any(np.isnan(X_train)) or np.any(np.isinf(X_train)):
                logging.warning("Training data contains NaN or inf values")
                return 0.5
            if np.any(np.isnan(X_val)) or np.any(np.isinf(X_val)):
                logging.warning("Validation data contains NaN or inf values")
                return 0.5
                
            model = LogisticRegression(**params)
            model.fit(X_train, y_train)
            
            preds = model.predict_proba(X_val)[:, 1]
            auc = roc_auc_score(y_val, preds)
            return auc
        except Exception as e:
            # Return low score if model fails to converge or other issues
            logging.warning(f"Trial failed: {e}")
            return 0.5

    study = optuna.create_study(direction='maximize', study_name='LogisticRegression_Mortality_Prediction')
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, timeout=OPTUNA_TIMEOUT)
    
    logging.info(f"✓ Optuna study complete. Best AUROC on validation set: {study.best_value:.4f}")
    logging.info("Best hyperparameters found:")
    for key, value in study.best_params.items():
        logging.info(f"  {key}: {value}")
        
    return study.best_params

# =============================================================================
# MAIN EXECUTION SCRIPT
# =============================================================================

def main():
    """Main function to run the entire Phase I analysis."""
    start_time = time.time()
    
    # --- 1. Load Data ---
    df_patients, df_ts_raw = load_data(HDF_FILE_PATH, n_samples=DRY_RUN_PATIENTS if DRY_RUN else None)
    
    # --- 2. Apply Time Window Filtering ---
    logging.info(f"Applying time window filtering: {WINDOW_SIZE}h window + {GAP_TIME}h gap...")
    
    # First, identify patients who have sufficient data (max_hours > WINDOW_SIZE + GAP_TIME)
    # Since hours_in is in the MultiIndex, we need to access it via the index
    patient_max_hours = df_ts_raw.groupby(level='icustay_id').apply(lambda x: x.index.get_level_values('hours_in').max())
    valid_patients = patient_max_hours[patient_max_hours > (WINDOW_SIZE + GAP_TIME)]
    logging.info(f"Patients with sufficient data: {len(valid_patients)} out of {len(patient_max_hours)}")
    
    # Filter patients to only those with sufficient data
    df_patients = df_patients[df_patients.index.get_level_values('icustay_id').isin(valid_patients.index)]
    logging.info(f"✓ Filtered patients: {df_patients.shape}")
    
    # Filter time-series data to the specified window (first WINDOW_SIZE hours only)
    df_ts_raw = df_ts_raw[
        (df_ts_raw.index.get_level_values('icustay_id').isin(valid_patients.index)) &
        (df_ts_raw.index.get_level_values('hours_in') < WINDOW_SIZE)
    ]
    logging.info(f"✓ Filtered time-series to {WINDOW_SIZE}h window: {df_ts_raw.shape}")
    
    # --- 3. Define Feature Sets ---
    available_cols = set(df_ts_raw.columns)
    dynamic_features_final = [f for f in DYNAMIC_FEATURES if f in available_cols]
    static_features_final = [f for f in available_cols if f not in dynamic_features_final]
    logging.info(f"Identified {len(dynamic_features_final)} dynamic and {len(static_features_final)} static features.")
    
    # --- 4. Split Data with Stratification to Ensure Balanced Class Distribution ---
    logging.info("Splitting data into train/validation/test sets by subject_id with stratification...")
    
    # Create a summary dataset with one row per subject to enable stratification
    subject_outcomes = df_patients.groupby('subject_id')[TARGET_VARIABLE].max()  # Take max outcome per subject
    subjects = subject_outcomes.index.values
    outcomes = subject_outcomes.values
    
    logging.info(f"Subject-level mortality rate: {outcomes.mean():.3f} ({outcomes.sum()}/{len(outcomes)})")
    
    # Use stratified split at subject level to ensure balanced class distribution
    from sklearn.model_selection import train_test_split
    
    # First split: train+val vs test (75% vs 25%)
    train_val_subjects, test_subjects, train_val_outcomes, test_outcomes = train_test_split(
        subjects, outcomes, test_size=0.25, random_state=SEED, stratify=outcomes
    )
    
    # Second split: train vs val (87.5% vs 12.5% of remaining, resulting in ~70% vs 10% overall)
    train_subjects, val_subjects, train_outcomes, val_outcomes = train_test_split(
        train_val_subjects, train_val_outcomes, test_size=0.125, random_state=SEED, stratify=train_val_outcomes
    )
    
    logging.info(f"Stratified split sizes:")
    logging.info(f"  Train: {len(train_subjects)} subjects (mortality: {train_outcomes.mean():.3f})")
    logging.info(f"  Val:   {len(val_subjects)} subjects (mortality: {val_outcomes.mean():.3f})")
    logging.info(f"  Test:  {len(test_subjects)} subjects (mortality: {test_outcomes.mean():.3f})")

    # Create subject-based masks to prevent data leakage (same subject in multiple splits)
    train_patient_mask = df_patients.index.get_level_values('subject_id').isin(train_subjects)
    val_patient_mask = df_patients.index.get_level_values('subject_id').isin(val_subjects)
    test_patient_mask = df_patients.index.get_level_values('subject_id').isin(test_subjects)
    
    # Get icustay_ids for each split (ensuring no subject appears in multiple splits)
    train_icustay_ids = df_patients[train_patient_mask].index.get_level_values('icustay_id').unique()
    val_icustay_ids = df_patients[val_patient_mask].index.get_level_values('icustay_id').unique()
    test_icustay_ids = df_patients[test_patient_mask].index.get_level_values('icustay_id').unique()
    
    # Verify no data leakage: check that no subject appears in multiple splits
    train_subjects_check = df_patients[train_patient_mask].index.get_level_values('subject_id').unique()
    val_subjects_check = df_patients[val_patient_mask].index.get_level_values('subject_id').unique()
    test_subjects_check = df_patients[test_patient_mask].index.get_level_values('subject_id').unique()
    
    # Check for overlaps
    train_val_overlap = set(train_subjects_check) & set(val_subjects_check)
    train_test_overlap = set(train_subjects_check) & set(test_subjects_check)
    val_test_overlap = set(val_subjects_check) & set(test_subjects_check)
    
    if train_val_overlap or train_test_overlap or val_test_overlap:
        logging.error(f"DATA LEAKAGE DETECTED!")
        logging.error(f"Train-Val overlap: {len(train_val_overlap)} subjects")
        logging.error(f"Train-Test overlap: {len(train_test_overlap)} subjects")
        logging.error(f"Val-Test overlap: {len(val_test_overlap)} subjects")
        raise ValueError("Data leakage: Same subjects found in multiple splits!")
    else:
        logging.info("✓ No data leakage: Subjects are properly separated across splits")
    
    # Log detailed split statistics
    logging.info(f"Final split statistics:")
    logging.info(f"  Train: {len(train_subjects_check)} subjects, {len(train_icustay_ids)} ICU stays")
    logging.info(f"  Val:   {len(val_subjects_check)} subjects, {len(val_icustay_ids)} ICU stays")
    logging.info(f"  Test:  {len(test_subjects_check)} subjects, {len(test_icustay_ids)} ICU stays")
    
    # --- 5. Check for Cached Preprocessing or Engineer Features ---
    cached_data = None
    if USE_CACHED_PREPROCESSING:
        cached_data = load_preprocessed_data()
    
    if cached_data is not None:
        # Use cached preprocessed data  
        X_train, X_val, X_test, y_train, y_val, y_test, scaler, imputation_values = cached_data
        logging.info("Using cached preprocessed data - skipping feature engineering!")
    else:
        # Perform feature engineering and preprocessing
        logging.info("No cached data found or caching disabled. Performing feature engineering...")
        
        # We engineer features on each data split separately to avoid any data leakage
        train_ts_mask = df_ts_raw.index.get_level_values('icustay_id').isin(train_icustay_ids)
        val_ts_mask = df_ts_raw.index.get_level_values('icustay_id').isin(val_icustay_ids)
        test_ts_mask = df_ts_raw.index.get_level_values('icustay_id').isin(test_icustay_ids)
        
        X_train = engineer_features(df_ts_raw[train_ts_mask], dynamic_features_final, static_features_final)
        X_val = engineer_features(df_ts_raw[val_ts_mask], dynamic_features_final, static_features_final)
        X_test = engineer_features(df_ts_raw[test_ts_mask], dynamic_features_final, static_features_final)

        # --- 6. Align Features and Handle Missing Values ---
        # Ensure all splits have the same columns, filling missing with 0
        logging.info("Aligning feature columns across splits and handling NaNs...")
        all_cols = X_train.columns.union(X_val.columns).union(X_test.columns)
        X_train = X_train.reindex(columns=all_cols, fill_value=0)
        X_val = X_val.reindex(columns=all_cols, fill_value=0)
        X_test = X_test.reindex(columns=all_cols, fill_value=0)

        # Impute remaining NaNs (e.g., from std of a single value) with the median of the training set
        imputation_values = X_train.median()
        X_train = X_train.fillna(imputation_values)
        X_val = X_val.fillna(imputation_values)
        X_test = X_test.fillna(imputation_values)
        
        # Scale features
        scaler = StandardScaler()
        X_train = pd.DataFrame(scaler.fit_transform(X_train), index=X_train.index, columns=X_train.columns)
        X_val = pd.DataFrame(scaler.transform(X_val), index=X_val.index, columns=X_val.columns)
        X_test = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)
        
        # --- 7. Prepare Target Variable ---
        # CRITICAL FIX: Align target variables with engineered feature indices
        # The engineered features have icustay_id as index, so we need to align targets properly
        y_train = df_patients.loc[df_patients.index.get_level_values('icustay_id').isin(X_train.index), TARGET_VARIABLE]
        y_train = y_train.groupby('icustay_id').first()  # Handle any duplicates
        y_train = y_train.reindex(X_train.index)  # Ensure perfect alignment
        
        y_val = df_patients.loc[df_patients.index.get_level_values('icustay_id').isin(X_val.index), TARGET_VARIABLE]
        y_val = y_val.groupby('icustay_id').first()
        y_val = y_val.reindex(X_val.index)
        
        y_test = df_patients.loc[df_patients.index.get_level_values('icustay_id').isin(X_test.index), TARGET_VARIABLE]
        y_test = y_test.groupby('icustay_id').first()
        y_test = y_test.reindex(X_test.index)
        
        # VERIFICATION: Check index alignment between X and y
        logging.info(f"Index alignment verification:")
        logging.info(f"  X_train.index matches y_train.index: {X_train.index.equals(y_train.index)}")
        logging.info(f"  X_val.index matches y_val.index: {X_val.index.equals(y_val.index)}")
        logging.info(f"  X_test.index matches y_test.index: {X_test.index.equals(y_test.index)}")
        
        # Debug: Check class distributions in each split
        logging.info(f"Target variable distributions after splitting:")
        logging.info(f"  Train: {dict(y_train.value_counts())} (mortality: {y_train.mean():.3f})")
        logging.info(f"  Val:   {dict(y_val.value_counts())} (mortality: {y_val.mean():.3f})")
        logging.info(f"  Test:  {dict(y_test.value_counts())} (mortality: {y_test.mean():.3f})")
        
        # Check for any misaligned indices or NaN targets
        if y_train.isna().any() or y_val.isna().any() or y_test.isna().any():
            logging.error("CRITICAL: Found NaN values in target variables after alignment!")
            logging.error(f"  y_train NaNs: {y_train.isna().sum()}")
            logging.error(f"  y_val NaNs: {y_val.isna().sum()}")
            logging.error(f"  y_test NaNs: {y_test.isna().sum()}")
            raise ValueError("Target variable alignment failed - contains NaN values!")
        
        # Check for single-class issues
        if y_val.nunique() == 1:
            logging.error(f"CRITICAL: Validation set has only one class! Cannot calculate ROC AUC.")
            logging.error(f"Consider increasing sample size or adjusting stratification.")
            raise ValueError("Validation set contains only one class - stratification failed!")
        
        # Save preprocessed data to cache
        if USE_CACHED_PREPROCESSING:
            save_preprocessed_data(X_train, X_val, X_test, y_train, y_val, y_test, scaler, imputation_values)
    
    logging.info(f"Data ready for modeling. Shapes: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")

    # --- 7.5. Additional Data Cleaning for Logistic Regression ---
    logging.info("Performing additional data cleaning for Logistic Regression...")
    
    # Check for problematic values
    logging.info(f"Before cleaning - NaN counts:")
    logging.info(f"  X_train: {X_train.isna().sum().sum()}")
    logging.info(f"  X_val: {X_val.isna().sum().sum()}")
    logging.info(f"  X_test: {X_test.isna().sum().sum()}")
    
    logging.info(f"Before cleaning - Inf counts:")
    logging.info(f"  X_train: {np.isinf(X_train).sum().sum()}")
    logging.info(f"  X_val: {np.isinf(X_val).sum().sum()}")
    logging.info(f"  X_test: {np.isinf(X_test).sum().sum()}")
    
    # Replace infinite values with NaN first, then handle all NaNs consistently
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_val = X_val.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.replace([np.inf, -np.inf], np.nan)
    
    # Fill remaining NaNs with column medians from training set
    for col in X_train.columns:
        if X_train[col].isna().any():
            median_val = X_train[col].median()
            if pd.isna(median_val):  # If median is also NaN, use 0
                median_val = 0.0
            X_train[col] = X_train[col].fillna(median_val)
            X_val[col] = X_val[col].fillna(median_val)
            X_test[col] = X_test[col].fillna(median_val)
    
    # Final verification
    logging.info(f"After cleaning - NaN counts:")
    logging.info(f"  X_train: {X_train.isna().sum().sum()}")
    logging.info(f"  X_val: {X_val.isna().sum().sum()}")
    logging.info(f"  X_test: {X_test.isna().sum().sum()}")
    
    logging.info(f"After cleaning - Inf counts:")
    logging.info(f"  X_train: {np.isinf(X_train).sum().sum()}")
    logging.info(f"  X_val: {np.isinf(X_val).sum().sum()}")
    logging.info(f"  X_test: {np.isinf(X_test).sum().sum()}")
    
    # Check for extremely large values that might cause overflow
    max_abs_vals = X_train.abs().max()
    large_value_cols = max_abs_vals[max_abs_vals > 1e10].index.tolist()
    if large_value_cols:
        logging.warning(f"Found {len(large_value_cols)} columns with very large values (>1e10)")
        logging.warning(f"Large value columns: {large_value_cols[:5]}...")  # Show first 5
        
        # Clip extremely large values
        for col in large_value_cols:
            p99 = X_train[col].quantile(0.99)
            p1 = X_train[col].quantile(0.01)
            X_train[col] = X_train[col].clip(p1, p99)
            X_val[col] = X_val[col].clip(p1, p99)
            X_test[col] = X_test[col].clip(p1, p99)
        
        logging.info(f"✓ Clipped extreme values using 1st-99th percentile bounds")

    # --- 8. Tune Hyperparameters ---
    best_params = tune_logistic_regression_with_optuna(X_train, y_train, X_val, y_val)
    
    # --- 9. Train Final Model & Evaluate ---
    logging.info("Training final model on combined train+validation data...")
    X_train_full = pd.concat([X_train, X_val])
    y_train_full = pd.concat([y_train, y_val])
    
    # Final data validation before model training
    logging.info("Final data validation before model training...")
    if X_train_full.isna().any().any():
        logging.error("CRITICAL: Training data still contains NaN values!")
        raise ValueError("Training data contains NaN values after cleaning!")
    
    if np.isinf(X_train_full).any().any():
        logging.error("CRITICAL: Training data still contains infinite values!")
        raise ValueError("Training data contains infinite values after cleaning!")
    
    # Calculate class weights for final model
    classes = np.unique(y_train_full)
    class_weights = compute_class_weight('balanced', classes=classes, y=y_train_full)
    final_class_weight_dict = {classes[i]: class_weights[i] for i in range(len(classes))}
    
    final_params = best_params.copy()
    final_params['class_weight'] = final_class_weight_dict
    final_params['random_state'] = SEED
    final_params['max_iter'] = 1000
    final_params['n_jobs'] = -1
    
    logging.info(f"Final model parameters: {final_params}")
    model = LogisticRegression(**final_params)
    model.fit(X_train_full, y_train_full)
    
    # --- 10. Final Evaluation on Test Set ---
    logging.info("--- FINAL EVALUATION ON TEST SET ---")
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    
    # Calculate comprehensive metrics with uncertainty quantification
    evaluation_results = evaluate_with_uncertainty(
        y_test.values, y_pred_proba, y_pred, n_bootstrap=1000
    )
    
    # Extract results
    auroc_results = evaluation_results['auroc']
    auprc_results = evaluation_results['auprc']
    
    # Log results with confidence intervals
    logging.info(f"Test Set AUROC: {auroc_results['point_estimate']:.4f} "
                f"(95% CI: {auroc_results['ci_lower']:.4f}-{auroc_results['ci_upper']:.4f})")
    logging.info(f"Test Set AUPRC: {auprc_results['point_estimate']:.4f} "
                f"(95% CI: {auprc_results['ci_lower']:.4f}-{auprc_results['ci_upper']:.4f})")
    
    # Log additional uncertainty statistics
    logging.info(f"AUROC Bootstrap Std: {auroc_results['std']:.4f}")
    logging.info(f"AUPRC Bootstrap Std: {auprc_results['std']:.4f}")
    logging.info(f"Bootstrap samples used: {auroc_results['n_bootstrap']}")
    
    logging.info("Classification Report:")
    report = classification_report(y_test, y_pred)
    print(report)
    logging.info("\n" + report)
    
    logging.info("Confusion Matrix:")
    cm = evaluation_results['basic_metrics']['confusion_matrix']
    print(cm)
    logging.info("\n" + str(cm))
    
    # --- 11. Save Artifacts ---
    logging.info("Saving all Phase I artifacts...")
    
    # Save final model
    model_path = os.path.join(OUTPUT_DIR, 'model_1_logistic_baseline.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    logging.info(f"✓ Model saved to: {model_path}")
    
    # Save results with uncertainty quantification
    results = {
        'model_name': 'Model 1 (Logistic Regression Baseline)',
        'target_variable': TARGET_VARIABLE,
        'test_auroc': auroc_results['point_estimate'],
        'test_auroc_ci_lower': auroc_results['ci_lower'],
        'test_auroc_ci_upper': auroc_results['ci_upper'],
        'test_auroc_std': auroc_results['std'],
        'test_auprc': auprc_results['point_estimate'],
        'test_auprc_ci_lower': auprc_results['ci_lower'],
        'test_auprc_ci_upper': auprc_results['ci_upper'],
        'test_auprc_std': auprc_results['std'],
        'bootstrap_samples': auroc_results['n_bootstrap'],
        'evaluation_results_full': evaluation_results,
        'classification_report': classification_report(y_test, y_pred, output_dict=True),
        'confusion_matrix': cm.tolist(),
        'best_hyperparameters': best_params
    }
    results_path = os.path.join(OUTPUT_DIR, 'results_model_1_logistic.pkl')
    with open(results_path, 'wb') as f:
        pickle.dump(results, f)
    logging.info(f"✓ Results dictionary saved to: {results_path}")
    
    # Save final engineered datasets
    X_test.to_csv(os.path.join(OUTPUT_DIR, 'X_test_engineered.csv'))
    y_test.to_csv(os.path.join(OUTPUT_DIR, 'y_test.csv'))
    logging.info(f"✓ Engineered test set saved for future use.")
    
    total_time = time.time() - start_time
    logging.info(f"--- Phase I script finished successfully in {total_time/60:.2f} minutes. ---")

if __name__ == "__main__":
    main()

/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-06-23 11:44:23,467 [INFO] - Starting data loading from: ../data/raw/all_hourly_data.h5
2025-06-23 11:44:23,658 [INFO] - NumExpr defaulting to 8 threads.
2025-06-23 11:44:23,938 [INFO] - Loading patient data...
2025-06-23 11:44:24,541 [INFO] - ✓ Patients loaded: (34472, 28)
2025-06-23 11:44:24,543 [INFO] - Loading full time-series data (vitals_labs_mean)...
2025-06-23 11:44:26,363 [INFO] - ✓ Full time-series loaded: (2200954, 104)
2025-06-23 11:44:26,365 [INFO] - Found MultiIndex, flattening to first level.
2025-06-23 11:44:29,697 [INFO] - ✓ Time-series processed: (2200954, 104)
2025-06-23 11:44:29,725 [INFO] - Applying time window filtering: 24h window + 6h gap...
2025-06-23 11:45:12,297 [INFO] - Patients wi

              precision    recall  f1-score   support

           0       0.97      0.82      0.88      5351
           1       0.33      0.76      0.46       635

    accuracy                           0.81      5986
   macro avg       0.65      0.79      0.67      5986
weighted avg       0.90      0.81      0.84      5986

[[4367  984]
 [ 152  483]]


/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/ipykernel_launcher.py:896: FutureWarning: The signature of `Series.to_csv` was aligned to that of `DataFrame.to_csv`, and argument 'header' will change its default value from False to True: please pass an explicit value to suppress this warning.
2025-06-23 12:05:08,977 [INFO] - ✓ Engineered test set saved for future use.
2025-06-23 12:05:08,978 [INFO] - --- Phase I script finished successfully in 20.76 minutes. ---
